In [1]:
# -*- coding: utf-8 -*-
# =============================================================================
# AI Agent Template（工具呼叫版）
# Model: Qwen2.5-3B-Instruct（可替換成其他 HuggingFace 模型）
# =============================================================================


# -----------------------------------------------------------------------------
# Cell 1：安裝套件（只需執行一次）
# -----------------------------------------------------------------------------

import os
os.system("pip install -q transformers accelerate sentencepiece")

0

In [2]:
# -----------------------------------------------------------------------------
# Cell 2：import + 載入模型（只需執行一次，不要重跑）
# -----------------------------------------------------------------------------

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
import io
import contextlib
import ast
import operator

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # 替換成任何 HuggingFace Chat 模型

def load_model(model_name: str = MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

tokenizer, model = load_model()
print("✅ 模型載入完成")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ 模型載入完成


In [3]:
# -----------------------------------------------------------------------------
# Cell 3：行為設定（可隨時修改重跑）
# -----------------------------------------------------------------------------

# FIX: MAX_NEW_TOKENS 從 200 提升至 512，避免 code 生成時被截斷
MAX_NEW_TOKENS = 512    # 每次生成的最大 token 數
DO_SAMPLE      = False  # True = 隨機生成（創意）；False = 確定性生成（精準）
MAX_CODE_RETRY = 3      # code 執行失敗最多重試幾次

In [4]:
# -----------------------------------------------------------------------------
# Cell 4：llm()
# -----------------------------------------------------------------------------

def llm(messages: list) -> str:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [5]:
# -----------------------------------------------------------------------------
# Cell 5：Tool — calculator
# FIX: 改用 ast 安全計算，取代直接 eval()
# -----------------------------------------------------------------------------

# 支援的運算子
_OPERATORS = {
    ast.Add:  operator.add,
    ast.Sub:  operator.sub,
    ast.Mult: operator.mul,
    ast.Div:  operator.truediv,
    ast.Pow:  operator.pow,
    ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        return _OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _OPERATORS:
        return _OPERATORS[type(node.op)](_safe_eval(node.operand))
    else:
        raise ValueError(f"不支援的運算：{ast.dump(node)}")

def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        result = _safe_eval(tree)
        return str(result)
    except Exception as e:
        return f"計算錯誤：{e}"

In [6]:
# -----------------------------------------------------------------------------
# Cell 6：Tool — code 執行（extract_code / execute_code / run_code_iteration / code_loop）
# -----------------------------------------------------------------------------

def extract_code(text: str) -> str | None:
    """從回覆中抽出 ```python ... ``` 的內容，沒有則回傳 None"""
    match = re.search(r"```python\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else None

def execute_code(code: str) -> tuple[bool, str]:
    """執行 Python code，回傳 (成功與否, 輸出或錯誤訊息)
    FIX: exec 環境加入 __builtins__，確保 import 等內建功能可正常使用
    """
    stdout_capture = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, {"__builtins__": __builtins__})
        output = stdout_capture.getvalue()
        return True, output if output else "（執行成功，無輸出）"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def run_code_iteration(messages: list, attempt: int) -> tuple[bool, str, str]:
    """單次 code 疊代：產生或修正 code，執行後回傳 (成功與否, reply, 執行結果)"""
    reply = llm(messages)
    code = extract_code(reply)
    if not code:
        return False, reply, "未找到 code"
    print(f"[Code 嘗試 {attempt}/{MAX_CODE_RETRY}]\n{code}")
    success, result = execute_code(code)
    print(f"[執行結果] {'✅' if success else '❌'} {result}")
    return success, reply, result

def code_loop(messages: list, task: str) -> str:
    """自動疊代修正模式：最多重試 MAX_CODE_RETRY 次"""
    messages = messages.copy()
    messages.append({"role": "user", "content": f"請用 ```python\n...\n``` 格式寫出以下程式：{task}"})
    last_reply, last_result = "", ""
    for attempt in range(1, MAX_CODE_RETRY + 1):
        success, last_reply, last_result = run_code_iteration(messages, attempt)
        if success or attempt >= MAX_CODE_RETRY:
            if success:
                code = extract_code(last_reply)
                return f"```python\n{code}\n```\n執行結果：{last_result}"
            return f"超過最大嘗試次數（{MAX_CODE_RETRY}），最後的錯誤：{last_result}"
        messages.append({"role": "assistant", "content": last_reply})
        messages.append({"role": "user",      "content": f"執行錯誤：{last_result}\n請修正 code，只輸出 ```python\n...\n``` 格式。"})
    return f"超過最大嘗試次數（{MAX_CODE_RETRY}），最後的錯誤：{last_result}"

In [7]:
# -----------------------------------------------------------------------------
# Cell 7：Tool 選擇器（TOOLS 註冊 + build_tool_prompt + parse_tool_call）
# -----------------------------------------------------------------------------

# ✏️ 新增 tool 後註冊進來，agent 會自動支援
TOOLS = {
    "CALCULATOR": calculator,
    # "CODE":        code_loop,
    # "WEB_SEARCH": web_search,
}

def build_tool_prompt(tools: dict) -> str:
    tool_lines = "\n".join(f"- {name}(參數)" for name in tools)
    # FIX: 修正規則編號跳號（原本 1、3、4，現在改為 1、2、3）
    return f"""你是一個 AI Agent，必須嚴格遵守以下規則：

規則：
1. 遇到數學計算，你【絕對不能】自己計算，必須呼叫工具。
2. 呼叫工具時，只能輸出一行：工具名稱(參數)，不能有其他文字。
3. 不需要工具時，才直接回答。

可用工具：
{tool_lines}

"""

def parse_tool_call(response: str, tools: dict):
    """逐行比對已註冊工具，回傳 (tool_name, args) 或 (None, None)"""
    for line in response.splitlines():
        line = line.strip()
        for name in tools:
            if line.startswith(f"{name}("):
                match = re.search(rf"{name}\((.+)\)", line)
                if match:
                    return name, match.group(1)
    return None, None

In [8]:
# -----------------------------------------------------------------------------
# Cell 8：agent()
# FIX 1: tool 結果餵回模型，讓模型整合後再回答
# FIX 2: 移除冗餘的 CODE 特判，統一透過 TOOLS dict 呼叫
# -----------------------------------------------------------------------------

def agent(messages: list) -> str:
    messages = messages.copy()
    response = llm(messages)
    print(f"[模型] {response.splitlines()[0].strip()}")

    tool_name, tool_args = parse_tool_call(response, TOOLS)
    if tool_name:
        print(f"[工具呼叫] {tool_name}({tool_args})")
        tool_result = TOOLS[tool_name](tool_args)
        print(f"[工具結果] {tool_result}")

        # FIX: 將工具結果加回對話，讓模型產生自然語言的最終回答
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user",      "content": f"工具執行結果：{tool_result}\n請根據此結果回答使用者。"})
        final_response = llm(messages)
        print(f"[最終回答] {final_response.splitlines()[0].strip()}")
        return final_response

    return response

In [9]:
# -----------------------------------------------------------------------------
# Cell 9：SYSTEM_PROMPT + history 初始化
# -----------------------------------------------------------------------------

SYSTEM_PROMPT = "你是一個 AI Agent。\n\n" + build_tool_prompt(TOOLS)

history = [{"role": "system", "content": SYSTEM_PROMPT}]
print("✅ History 初始化完成")

✅ History 初始化完成


In [ ]:
# -----------------------------------------------------------------------------
# Cell 10：多輪對話（輸入 clear 重置對話，exit 結束）
# FIX: history 正確記錄工具呼叫的完整脈絡（assistant 呼叫 + 工具結果 + 最終回答）
# -----------------------------------------------------------------------------

print("開始對話（clear 重置對話／exit 結束）")
while True:
    USER_PROMPT = input("你：").strip()

    if USER_PROMPT.lower() == "exit":
        print("結束對話")
        break

    if USER_PROMPT.lower() == "clear":
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("🔄 對話已重置\n")
        continue

    history.append({"role": "user", "content": USER_PROMPT})

    # FIX: 在 agent() 外部重建工具呼叫脈絡，確保 history 完整記錄
    messages_snapshot = history.copy()
    response = llm(messages_snapshot)

    tool_name, tool_args = parse_tool_call(response, TOOLS)
    if tool_name:
        print(f"[工具呼叫] {tool_name}({tool_args})")
        tool_result = TOOLS[tool_name](tool_args)
        print(f"[工具結果] {tool_result}")

        # 將工具呼叫過程完整寫入 history
        history.append({"role": "assistant", "content": response})
        history.append({"role": "user",      "content": f"工具執行結果：{tool_result}\n請根據此結果回答使用者。"})
        reply = llm(history)
        history.append({"role": "assistant", "content": reply})
    else:
        reply = response
        history.append({"role": "assistant", "content": reply})

    print(f"Agent：{reply}\n")

開始對話（clear 重置對話／exit 結束）
你：計算 5 * 10 + 62 = 


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[工具呼叫] CALCULATOR(5*10+62)
[工具結果] 112
Agent：112

你：今天天氣如何
Agent：請提供您所在的城市名稱，我才能幫您查詢天氣。

你：台北
Agent：查詢天氣(台北)

